In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import requests


BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR
OUTPUT_DIR = BASE_DIR / "output"
BUDGET_FILE = BASE_DIR / "master" / "budget.csv"

TODAY = datetime.today()

# --------------------
# 決算期（8月末）
# --------------------
def fiscal_start(date):
    year = date.year if date.month >= 9 else date.year - 1
    return datetime(year, 9, 1)

def fiscal_end(date):
    year = date.year + 1 if date.month >= 9 else date.year
    return datetime(year, 8, 31)

# --------------------
# CSV読込（支店名はファイル名）
# --------------------
def load_sales():
    dfs = []
    files = list(INPUT_DIR.glob("*.csv"))

    if not files:
        raise FileNotFoundError("inputフォルダにCSVが見つかりません")

    print("読み込み対象ファイル:")
    for f in files:
        print(" -", f.name)

    for file in files:
        df = pd.read_csv(file, encoding="cp932")

        # 列存在チェック（事故防止）
        required_cols = ["請求日", "差引請求額", "担当部署名"]
        for col in required_cols:
            if col not in df.columns:
                raise KeyError(f"{file.name} に列 '{col}' がありません")

        df["請求日"] = pd.to_datetime(df["請求日"])
        df["支店"] = df["担当部署名"]
        df["売上金額"] = df["差引請求額"]

        dfs.append(df[["請求日", "支店", "売上金額"]])

    return pd.concat(dfs, ignore_index=True)


# --------------------
# 集計
# --------------------
def aggregate(df):
    start = fiscal_start(TODAY)
    last_month_end = TODAY.replace(day=1) - pd.Timedelta(days=1)
    last_month_start = last_month_end.replace(day=1)

    fiscal_df = df[(df["請求日"] >= start) & (df["請求日"] <= last_month_end)]

    monthly = fiscal_df[
        (fiscal_df["請求日"] >= last_month_start)
    ].groupby("支店")["売上金額"].sum().rename("前月売上")

    cumulative = fiscal_df.groupby("支店")["売上金額"].sum().rename("期累計売上")

    return pd.concat([monthly, cumulative], axis=1).fillna(0)

# --------------------
# 予算・着地見込み
# --------------------
def add_budget_forecast(df):
    budget = pd.read_csv(BUDGET_FILE).set_index("支店")
    df = df.join(budget)

    months_passed = ((TODAY.year - fiscal_start(TODAY).year) * 12 +
                     TODAY.month - 9) % 12 + 1

    df["着地見込み"] = (df["期累計売上"] / months_passed) * 12
    df["不足額"] = df["期予算"] - df["期累計売上"]

    return df

# --------------------
# Google Chat通知
# --------------------
def notify_chat(file_url):
    message = {
        "text": (
            "📊 月次売上レポート（前月実績・期進捗）\n\n"
            "・前月売上\n"
            "・期累計売上\n"
            "・予算差異\n"
            "・着地見込み\n\n"
            f"▶ CSVはこちら\n{file_url}"
        )
    }
    requests.post(config.CHAT_WEBHOOK_URL, json=message)

# --------------------
# メイン
# --------------------
def main():
    sales = load_sales()
    summary = aggregate(sales)
    summary = add_budget_forecast(summary)

    summary.loc["合計"] = summary.sum(numeric_only=True)

    OUTPUT_DIR.mkdir(exist_ok=True)
    filename = OUTPUT_DIR / f"sales_report_{TODAY.strftime('%Y%m')}.csv"
    summary.to_csv(filename, encoding="utf-8-sig")

    # 手動アップロード or Drive API連携可
    notify_chat("（ここにDriveのCSVリンク）")

    print("✅ 月次レポート生成・通知完了")

if __name__ == "__main__":
    main()


SyntaxError: invalid character '（' (U+FF08) (347017759.py, line 8)

In [13]:
print("BASE_DIR:", BASE_DIR)
print("INPUT_DIR exists:", INPUT_DIR.exists())
print("CSV files:", list(INPUT_DIR.glob("*.csv")))


BASE_DIR: c:\Users\tsuji\OneDrive\Desktop\lesson\python-basic-kadai\python-excel-kadai\work_flow
INPUT_DIR exists: False
CSV files: []
